# Architecture analysis: search log vs NB201 ground truth

For one search run, join three accuracy signals per architecture and report Kendall tau + Spearman rho on each pair.

**Inputs**
- `LOG_PATH` — `log.txt` from a search run.
- `LOOKUP_PATH` — NB201 ground-truth catalog (`arch_str` -> final accuracy, 0-1).

**Primary comparisons**
1. `pred_acc` vs `lookup_acc` — is the predictor any good in absolute terms?
2. `valid_acc` vs `lookup_acc` — is the 20-epoch proxy training faithful?
3. `pred_acc` vs `valid_acc` — for reference; same number `compute_run_metrics` reports.

Suggested diagnostics below (scatter plots, top-N overlap, residual histograms).

In [ ]:
import json
import re
import sys
from pathlib import Path

# Make the project root importable so `misc.*` / `nap2.*` resolve.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from misc.log_summary import scrape
from nap2.training.evaluate import compute_metrics

# --- Paste your paths here ---
LOG_PATH    = Path("PASTE/PATH/TO/log.txt")
# In-repo default catalog: PROJECT_ROOT / "nap2" / "nap2_log_snap1_cifar10.json"
LOOKUP_PATH = Path("PASTE/PATH/TO/lookup.json")

In [ ]:
data = scrape(LOG_PATH)
with open(LOOKUP_PATH) as f:
    lookup = json.load(f)

len(data), len(lookup)

## Join: build three parallel dicts keyed by arch_id

Per arch: pull `arch_str` out of the genotype repr, look it up in the catalog. An arch contributes to `lookup_d` only if its `arch_str` is a catalog key; to `pred_d` only if `pred_acc` is non-None.

Coverage line at the bottom catches a GA producing arch_strs outside the catalog or a predictor that crashed on some archs.

In [ ]:
_ARCH_STR_RE = re.compile(r"arch_str=['\"]([^'\"]+)['\"]")

def _arch_str(genotype_str):
    if not genotype_str:
        return None
    m = _ARCH_STR_RE.search(genotype_str)
    return m.group(1) if m else None

pred_d, valid_d, lookup_d = {}, {}, {}
n_missing_arch_str = n_missing_in_lookup = n_pred_failed = 0

for arch_id, row in data.items():
    pred  = row.get("pred_acc")
    valid = row.get("valid_acc")
    arch_str = _arch_str(row.get("genotype"))

    if valid is not None:
        valid_d[arch_id] = float(valid)
    if pred is None:
        n_pred_failed += 1
    else:
        pred_d[arch_id] = float(pred)

    if arch_str is None:
        n_missing_arch_str += 1
    elif arch_str in lookup:
        lookup_d[arch_id] = float(lookup[arch_str])
    else:
        n_missing_in_lookup += 1

print(f"total scraped              : {len(data)}")
print(f"with valid_acc             : {len(valid_d)}")
print(f"with pred_acc              : {len(pred_d)}  (failures: {n_pred_failed})")
print(f"with lookup_acc            : {len(lookup_d)}")
print(f"missing genotype/arch_str  : {n_missing_arch_str}")
print(f"arch_str not in lookup     : {n_missing_in_lookup}")

## 1. `pred_acc` vs `lookup_acc` (predictor quality vs ground truth)

In [ ]:
compute_metrics(pred_d, lookup_d)

## 2. `valid_acc` vs `lookup_acc` (proxy quality vs ground truth)

In [ ]:
compute_metrics(valid_d, lookup_d)

## 3. `pred_acc` vs `valid_acc` (reference — same number `compute_run_metrics` reports)

In [ ]:
compute_metrics(pred_d, valid_d)

## Diagnostic: scatter plots

Three panels, one per pair, with the y=x reference line. Catches what a single correlation number hides — a flat predictor (output range collapsed), U-shaped error, or a few outliers driving the correlation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _paired(a, b):
    keys = sorted(set(a) & set(b))
    return np.array([a[k] for k in keys]), np.array([b[k] for k in keys])

pairs = [
    ("pred_acc",  "lookup_acc", pred_d,  lookup_d),
    ("valid_acc", "lookup_acc", valid_d, lookup_d),
    ("pred_acc",  "valid_acc",  pred_d,  valid_d),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (xn, yn, xd, yd) in zip(axes, pairs):
    x, y = _paired(xd, yd)
    ax.scatter(x, y, alpha=0.5, s=12)
    lo = float(min(x.min(), y.min())) if len(x) else 0.0
    hi = float(max(x.max(), y.max())) if len(x) else 1.0
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, alpha=0.4)
    ax.set_xlabel(xn)
    ax.set_ylabel(yn)
    ax.set_title(f"{xn} vs {yn}  (n={len(x)})")
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Diagnostic: top-N agreement

How many of the top-N by predictor (or proxy) are also in the top-N by ground truth? For NAS we care about picking winners, so this can look great even when KT is mediocre — or vice versa.

In [ ]:
def topn_overlap(score_d, truth_d, n):
    keys = sorted(set(score_d) & set(truth_d))
    if len(keys) < n:
        return None
    top_by_score = set(sorted(keys, key=lambda k: -score_d[k])[:n])
    top_by_truth = set(sorted(keys, key=lambda k: -truth_d[k])[:n])
    return len(top_by_score & top_by_truth) / n

for n in (5, 10, 20):
    p = topn_overlap(pred_d, lookup_d, n)
    v = topn_overlap(valid_d, lookup_d, n)
    p_s = f"{p:.2f}" if p is not None else " n/a"
    v_s = f"{v:.2f}" if v is not None else " n/a"
    print(f"top-{n:>2}  pred vs lookup = {p_s}    valid vs lookup = {v_s}")

## Diagnostic: residual distributions

Histograms of `pred_acc - lookup_acc` and `valid_acc - lookup_acc`. Spots systematic bias — e.g. a predictor that consistently over- or under-shoots, or a proxy that plateaus at ~70% regardless of true ability.

In [ ]:
def _residuals(score_d, truth_d):
    keys = sorted(set(score_d) & set(truth_d))
    return np.array([score_d[k] - truth_d[k] for k in keys])

r_pred  = _residuals(pred_d,  lookup_d)
r_valid = _residuals(valid_d, lookup_d)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, r, title in zip(axes, (r_pred, r_valid),
                        ("pred_acc - lookup_acc", "valid_acc - lookup_acc")):
    ax.hist(r, bins=30, alpha=0.7)
    ax.axvline(0, color="k", lw=0.8, alpha=0.5)
    if len(r):
        ax.axvline(r.mean(), color="r", lw=1, alpha=0.7, label=f"mean={r.mean():.3f}")
        ax.legend()
    ax.set_xlabel(title)
    ax.set_title(f"{title}  (n={len(r)})")
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()